In [ ]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

/home/mariam/miniconda3/envs/paper_recommender/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("../data/arxiv_cleaned.csv")

In [3]:
df.shape

(41127, 6)

In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1382.02it/s]


In [6]:
embeddings = np.load("../models/transformer_embeddings.npy")

embeddings.shape

(41127, 384)

In [7]:
query = "deep learning for medical image classification"

In [10]:
query_embidding = model.encode([query])

query_embidding.shape

(1, 384)

In [16]:
similarity_scores=cosine_similarity(
    query_embidding,
    embeddings
).flatten()

In [17]:
similarity_scores.shape

(41127,)

In [18]:
top_indices=similarity_scores.argsort()[::-1][:10]

In [19]:
top_indices



array([18933, 12643, 27766, 12338, 15689, 27209,  1629,   139, 20715,
       12558])

In [21]:
recommendations=df.iloc[top_indices][['titles', 'abstracts']].copy()
recommendations["similarity_score"]=similarity_scores[top_indices]
recommendations

,titles,abstracts,similarity_score
18933,A Survey on Deep Learning in Medical Image Ana...,"Deep learning algorithms, in particular convol...",0.805474
12643,Deep Learning for Medical Image Processing: Ov...,Healthcare sector is totally different from ot...,0.763960
27766,Going Deep in Medical Image Analysis: Concepts...,Medical Image Analysis is currently experienci...,0.736035
12338,A Gentle Introduction to Deep Learning in Medi...,This paper tries to give a gentle introduction...,0.728671
15689,Medical Image Analysis using Convolutional Neu...,The science of solving clinical problems by an...,0.718911
27209,Explainable deep learning models in medical im...,Deep learning methods have been very effective...,0.697100
1629,An overview of deep learning in medical imagin...,"What has happened in machine learning lately, ...",0.693737
139,Deep Learning in Cardiology,The medical field is creating large amount of ...,0.686712
20715,Deep Convolutional Neural Networks for Compute...,Remarkable progress has been made in image rec...,0.673728
12558,DLTK: State of the Art Reference Implementatio...,"We present DLTK, a toolkit providing baseline ...",0.658235


In [28]:
def recommend_from_query(query, top_k=10): # Validate query 
    if not isinstance(query, str) or not query.strip():
         raise ValueError( "Query must be a non-empty string." )
    
    
    query_embedding = model.encode([query])
    
    similarity_scores = cosine_similarity(
        query_embedding,
        embeddings
    ).flatten()
    
    top_indices = similarity_scores.argsort()[::-1][:top_k]
    
    recommendations = df.iloc[top_indices][
        ['titles', 'abstracts','terms']
    ].copy()
    
    recommendations['similarity_score'] = similarity_scores[top_indices]
    
    return recommendations

In [30]:
recommend_from_query(
    "transformers for natural language processing",
    top_k=5
)

,titles,abstracts,terms,similarity_score
574,Do Transformer Modifications Transfer Across I...,The research community has proposed copious mo...,"['cs.LG', 'cs.CL']",0.680657
788,Compressing Large-Scale Transformer-Based Mode...,Pre-trained Transformer-based models have achi...,"['cs.LG', 'stat.ML']",0.646041
865,Injecting Hierarchy with U-Net Transformers,The Transformer architecture has become increa...,"['cs.LG', 'cs.CL', 'stat.ML']",0.634418
21465,Complexity-based partitioning of CSFI problem ...,"In this paper, we propose a two-steps approach...",['cs.LG'],0.629947
725,Empirical Study of Transformers for Source Code,Initially developed for natural language proce...,"['cs.LG', 'cs.CL', 'cs.SE']",0.622828


In [63]:
import importlib
import src.recommendation

importlib.reload(src.recommendation)

from src.recommendation import (
    recommend_from_query,
    get_paper_by_rank
)

In [62]:
from src.recommendation import recommend_from_query

In [64]:
recommendations = recommend_from_query(
    "transformers for natural language processing",
    top_k=5
)

In [65]:
paper = get_paper_by_rank(
    recommendations,
    rank=2
)

In [66]:
paper["abstracts"]

'Pre-trained Transformer-based models have achieved state-of-the-art\nperformance for various Natural Language Processing (NLP) tasks. However, these\nmodels often have billions of parameters, and, thus, are too resource-hungry\nand computation-intensive to suit low-capability devices or applications with\nstrict latency requirements. One potential remedy for this is model\ncompression, which has attracted a lot of research attention. Here, we\nsummarize the research in compressing Transformers, focusing on the especially\npopular BERT model. In particular, we survey the state of the art in\ncompression for BERT, we clarify the current best practices for compressing\nlarge-scale Transformer models, and we provide insights into the workings of\nvarious methods. Our categorization and analysis also shed light on promising\nfuture research directions for achieving lightweight, accurate, and generic NLP\nmodels.'

In [67]:
paper

titles              Compressing Large-Scale Transformer-Based Mode...
abstracts           Pre-trained Transformer-based models have achi...
terms                                            ['cs.LG', 'stat.ML']
rank                                                                2
similarity_score                                             0.646041
Name: 788, dtype: object

In [57]:
recommend_from_query(
    "transformers for natural language processing",
    top_k=0
)

ValueError: top_k must be greater than 0.

In [50]:
recommend_from_query(
    "transformers for natural language processing",
    top_k=-5
)

ValueError: top_k must be greater than 0.

In [53]:
recommend_from_query(
    "transformers",
    top_k=5.5
)

ValueError: top_k must be an integer.

In [71]:
import importlib
import src.translation

importlib.reload(src.translation)

from src.translation import translate_abstract

ImportError: 
MarianTokenizer requires the SentencePiece library but it was not found in your environment. Check out the instructions on the
installation page of its repo: https://github.com/google/sentencepiece#installation and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.
